In [ ]:
%pip install -q timm
import json, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import torch, torch.nn as nn
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

In [ ]:
# датасет лежит рядом с проектом: data/dataset (см. README)
DS = Path("data/dataset")
lab = pd.read_csv(DS / "photos" / "metadata.csv")
lab = lab[~lab.label_conflict]
lab = lab.rename(columns={"dup_group": "group"})[["file", "cls", "group"]]

CLASSES = ["рядовая", "труднообогатимая", "оталькованная"]
lab["y"] = lab.cls.map({c: i for i, c in enumerate(CLASSES)})
lab = lab.dropna(subset=["y"]).reset_index(drop=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device, len(lab), lab.cls.value_counts().to_dict())

In [ ]:
# групповой сплит: дубликаты и фото одного образца не разъезжаются
rng = np.random.RandomState(0)
groups = sorted(lab.group.unique())
val_g = set(rng.choice(groups, int(len(groups) * 0.2), replace=False))
tr_df = lab[~lab.group.isin(val_g)].reset_index(drop=True)
va_df = lab[lab.group.isin(val_g)].reset_index(drop=True)
print(f"train {len(tr_df)} / val {len(va_df)}")

SZ = 512
NORM = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# аугментации: масштаб 80-500 мкм, освещение/контраст, расфокус, повороты
train_tf = T.Compose([
    T.RandomResizedCrop(SZ, scale=(0.08, 0.9)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.Lambda(lambda im: im.rotate(random.choice((0, 90, 180, 270)))),
    T.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.3, hue=0.05),
    T.RandomApply([T.GaussianBlur(7, sigma=(0.1, 2.0))], p=0.3),
    T.ToTensor(), NORM,
])
val_tf = T.Compose([T.Resize(SZ + 64), T.CenterCrop(SZ), T.ToTensor(), NORM])

class PhotoDS(Dataset):
    def __init__(self, df, tf):
        self.df, self.tf = df, tf
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(DS / r.file).convert("RGB")
        return self.tf(img), int(r.y)

tr_dl = DataLoader(PhotoDS(tr_df, train_tf), batch_size=64, shuffle=True,
                   num_workers=8, pin_memory=True)
va_dl = DataLoader(PhotoDS(va_df, val_tf), batch_size=64, num_workers=8)

In [ ]:
model = timm.create_model("convnext_tiny", pretrained=True, num_classes=3).to(device)
w = 1.0 / lab.y.value_counts(normalize=True).sort_index().values
crit = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=device),
                           label_smoothing=0.05)
opt = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
EPOCHS = 15
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=device == "cuda")

def evaluate():
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, y in va_dl:
            with torch.amp.autocast("cuda", enabled=device == "cuda"):
                out = model(x.to(device))
            ps.append(torch.softmax(out.float(), 1).cpu().numpy())
            ys.append(y.numpy())
    model.train()
    return np.concatenate(ys), np.concatenate(ps)

best_f1, log = 0.0, []
for ep in range(EPOCHS):
    t0, tot = time.time(), 0.0
    for x, y in tr_dl:
        with torch.amp.autocast("cuda", enabled=device == "cuda"):
            loss = crit(model(x.to(device)), y.to(device))
        opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        tot += float(loss) * len(x)
    sched.step()
    ys, ps = evaluate()
    f1 = f1_score(ys, ps.argmax(1), average="macro")
    star = ""
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), "cnn_classifier.pt")
        star = " <-- saved"
    print(f"ep {ep:02d} loss {tot/len(tr_df):.3f} macro-F1 {f1:.3f} "
          f"[{time.time()-t0:.0f}с]{star}", flush=True)
    log.append(dict(epoch=ep, loss=tot / len(tr_df), f1=float(f1)))

In [ ]:
ys, ps = evaluate()
pred = ps.argmax(1)
print(classification_report(ys, pred, target_names=CLASSES))
print(confusion_matrix(ys, pred))
auc = roc_auc_score(ys, ps, multi_class="ovr")
json.dump(dict(best_macro_f1=best_f1, auc_ovr=auc, classes=CLASSES,
               arch="convnext_tiny", size=SZ, log=log),
          open("cnn_metrics.json", "w"), ensure_ascii=False, indent=2)
# результат: cnn_classifier.pt, cnn_metrics.json -> artifacts/cnn/
print(f"best macro-F1 = {best_f1:.3f}")